# EuRoC Chebyshev Spectrogram

Fit spectral Chebyshev polynomials to fixed-duration intervals in a merged EuRoC CSV.

This notebook is intentionally thin: fitting, diagnostics, derived IMU norm signals, and Plotly figure builders live in `imuFactors.chebyshev_spectrogram`. The fit uses `gtsam.Chebyshev1Basis`, so `N` is the number of weighted spectral basis functions and the polynomial degree is `n = N - 1`. It does not use the `Chebyshev2` pseudo-spectral node-value parameterization.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import Any

from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as exc:
    HAS_WIDGETS = False
    WIDGET_IMPORT_ERROR = exc

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python" / "imuFactors").exists():
    REPO_ROOT = Path("/Users/dellaert/git/imuFactors")

PYTHON_DIR = REPO_ROOT / "python"
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

from imuFactors.chebyshev_spectrogram import (
    DEFAULT_SIGNAL_GROUP,
    SIGNAL_GROUPS,
    characteristic_windows,
    discover_euroc_files,
    fit_spectral_chebyshev_windows,
    interval_metrics_table,
    plot_average_spectra,
    plot_average_spectrogram_parts,
    plot_coefficient_spectrogram,
    plot_interval_coefficients,
    plot_interval_fit,
    plot_window_characteristics,
    summary_table,
    WINDOW_SECONDS,
)

DATA_DIR = REPO_ROOT / "data" / "euroc"
DATA_FILES = discover_euroc_files(DATA_DIR)
if not DATA_FILES:
    raise FileNotFoundError(f"No EuRoC CSV files found in {DATA_DIR}")

DEFAULT_FILE = DATA_DIR / "euroc_MH01.csv"
if not DEFAULT_FILE.exists():
    DEFAULT_FILE = DATA_FILES[0]

DEFAULT_N = 16
DEFAULT_WINDOW_SECONDS = WINDOW_SECONDS
print(f"Found {len(DATA_FILES)} EuRoC CSV files in {DATA_DIR}")

In [ ]:
def run_dashboard(
    path: str | Path = DEFAULT_FILE,
    coefficient_count: int = DEFAULT_N,
    signal_group: str = DEFAULT_SIGNAL_GROUP,
    window_seconds: float = DEFAULT_WINDOW_SECONDS,
    max_components: int = 6,
):
    """Fit a file and render the standard spectrogram views."""
    result = fit_spectral_chebyshev_windows(
        path,
        coefficient_count=int(coefficient_count),
        signal_group=signal_group,
        window_seconds=float(window_seconds),
    )
    selected = characteristic_windows(result)

    display(summary_table(result))
    display(interval_metrics_table(result, selected))
    plot_window_characteristics(result, selected).show()

    for label, window_index in selected.items():
        plot_interval_fit(
            result,
            window_index,
            label=label,
            max_components=max_components,
        ).show()
        plot_interval_coefficients(result, window_index, label=label).show()

    plot_coefficient_spectrogram(result).show()
    plot_average_spectra(result).show()
    plot_average_spectrogram_parts(result).show()
    return result

## Interactive Controls

Choose a merged EuRoC CSV, the spectral coefficient count `N`, the interval length, and a signal group. The `imu_norms` group fits the two derived auxiliary signals `[gyro_norm, accel_norm]`. At 200 Hz, a 1.0-second closed interval contains 201 samples and adjacent intervals share one endpoint.

In [ ]:
def make_dashboard_controls() -> tuple[Any, Any] | tuple[None, None]:
    if not HAS_WIDGETS:
        print("ipywidgets is not available; edit DEFAULT_FILE, DEFAULT_N, and DEFAULT_SIGNAL_GROUP manually.")
        print(f"Widget import error: {WIDGET_IMPORT_ERROR!r}")
        return None, None

    file_dropdown = widgets.Dropdown(
        options=[(path.name, str(path)) for path in DATA_FILES],
        value=str(DEFAULT_FILE),
        description="file",
        layout=widgets.Layout(width="560px"),
    )
    n_slider = widgets.IntSlider(
        value=DEFAULT_N,
        min=2,
        max=80,
        step=1,
        description="N",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    interval_slider = widgets.FloatSlider(
        value=DEFAULT_WINDOW_SECONDS,
        min=0.5,
        max=5.0,
        step=0.25,
        description="interval s",
        continuous_update=False,
        readout_format=".2f",
        layout=widgets.Layout(width="560px"),
    )
    group_dropdown = widgets.Dropdown(
        options=list(SIGNAL_GROUPS.keys()),
        value=DEFAULT_SIGNAL_GROUP,
        description="signals",
        layout=widgets.Layout(width="560px"),
    )
    component_slider = widgets.IntSlider(
        value=6,
        min=1,
        max=12,
        step=1,
        description="plot dims",
        continuous_update=False,
        layout=widgets.Layout(width="560px"),
    )
    run_button = widgets.Button(description="Run fit", button_style="primary", icon="play")
    output = widgets.Output()

    def on_run(_: Any) -> None:
        with output:
            clear_output(wait=True)
            global LAST_RESULT
            LAST_RESULT = run_dashboard(
                Path(file_dropdown.value),
                coefficient_count=int(n_slider.value),
                signal_group=str(group_dropdown.value),
                window_seconds=float(interval_slider.value),
                max_components=int(component_slider.value),
            )

    run_button.on_click(on_run)
    controls = widgets.VBox([
        widgets.HBox([file_dropdown]),
        widgets.HBox([n_slider]),
        widgets.HBox([interval_slider]),
        widgets.HBox([group_dropdown]),
        widgets.HBox([component_slider, run_button]),
    ])
    return controls, output


controls, output = make_dashboard_controls()
if controls is not None:
    display(controls, output)
    with output:
        LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_SIGNAL_GROUP)
else:
    LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_SIGNAL_GROUP)

## Working With Results

`LAST_RESULT.coeffs` has shape `(m, N, d)`: one row per interval, one spectral coefficient per Chebyshev basis function, and one selected signal component. `LAST_RESULT.coeff_energy` is the standardized `m x N` image used for the spectrogram.

Raw coefficient heatmaps show the actual least-squares weights multiplying `T_k(tau)`. Spectrogram and average spectra use robust component scaling so signals with different units can share one image. Derived auxiliary signals are available as `gyro_norm` and `accel_norm` through the `imu_norms` signal group.